# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring/ranking task, not classification or clustering. The output I need isn't a hard yes/no label on a page — it's a continuous priority score (or rank) across all candidate pages, because the real decision is "which pages go to the top of a limited-capacity queue this sprint," not "is this page good or bad." Ranking also matches how the pipeline's own metric works , and it degrades gracefully: even an imperfect model still orders pages better than a coin flip, whereas a binary classifier forces an arbitrary cutoff before I've earned the right to pick one

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Ideally I'd predict an observed outcome — e.g., whether a page's traffic/clicks measurably improved after it was refreshed — but that requires a before/after refresh history I likely don't have in this anonymized snapshot. So realistically I'll use a defined proxy: a rule-based composite built from things like days_since_update, search_volume, and clicks_90d (e.g., high demand + stale + underperforming = high proxy priority). I need to be upfront that this proxy encodes my own assumptions about what "needs a refresh" means — it's a stand-in for the real, unobserved outcome (did refreshing it actually help), not the outcome itself.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages my model ranks as highest priority, what fraction would a human reviewer actually agree deserve a refresh (checked against the proxy label, or a manual spot-check sample). I'm choosing this over overall accuracy or AUC because the real-world action only ever touches the top of the list — nobody works through all 30,000 rows — so what matters is whether the front of the queue is trustworthy, not how well-calibrated the model is on pages nobody will ever look at.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import os

LOCAL_PATH = "data/raw/content_refresh_anonymized.csv"
REMOTE_PATH = ("https://raw.githubusercontent.com/flyrank-bih/"
               "flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(LOCAL_PATH) if os.path.exists(LOCAL_PATH) else pd.read_csv(REMOTE_PATH)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(df.columns.tolist())

# One row = one page. Show a slice of the columns relevant to this lane.
cols = [c for c in ["days_since_update", "search_volume", "clicks_90d", "position_avg"] if c in df.columns]
display(df[cols].head(10)) if cols else display(df.head(10))

# Sketch what the proxy target column would look like (not a final version)
if {"days_since_update", "search_volume", "clicks_90d"}.issubset(df.columns):
    df["refresh_priority_proxy"] = (
        (df["days_since_update"] > df["days_since_update"].median()).astype(int)
        + (df["search_volume"] > df["search_volume"].median()).astype(int)
        + (df["clicks_90d"] < df["clicks_90d"].median()).astype(int)
    )
    print("\nSketch of proxy target distribution (0-3 scale):")
    print(df["refresh_priority_proxy"].value_counts().sort_index())

Rows: 30,000 | Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,search_volume,clicks_90d
0,10.0,29
1,90.0,7
2,0.0,11
3,10.0,58
4,0.0,24
5,720.0,1
6,0.0,0
7,590.0,1
8,0.0,29
9,0.0,2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement rule (e.g., "flag any page untouched for 180+ days with above-median search volume") treats every signal as equally important and independent, but in practice these factors interact and trade off in ways a few hand-tuned thresholds can't capture — a page that's slightly stale but has huge demand might matter more than a very stale page nobody searches for. A learned model can weigh dozens of signals together, find nonlinear combinations, and get recalibrated as new data comes in, instead of me guessing the "right" cutoffs by eye. Notebook 1 already demonstrated this concretely: the hand-written rule got roughly 24% right, while a learned model beat it by about 3x on the same data — the same gap I'd expect here since the underlying problem (many weak, interacting signals) is structurally similar.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.